# Environment

In [1]:
import sys
from pathlib import Path
import sqlite3
module_path = Path("../01_Functions_classes_and_variables").resolve()
sys.path.append(str(module_path))

from simulation_data_functions import *
from data_exploration_functions import *
from data_operations_functions import *

import matplotlib.pyplot as plt
from spreg import ML_Lag
import statsmodels.api as sm
from sklearn.ensemble import RandomForestRegressor
from econml.dml import CausalForestDML

from esda.moran import Moran


from sklearn.ensemble import RandomForestRegressor
from causalml.inference.meta import BaseSRegressor
from econml.dml import DML, LinearDML, SparseLinearDML, CausalForestDML
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.preprocessing import PolynomialFeatures

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor

In [2]:
repo_path_wndws = Path("../").resolve()
repo_path_unix = repo_path_wndws.as_posix() 
repo_path_r_string = repo_path_unix.replace("/", '\\')

# Data

In [3]:
df_names_list = [
    'gdf_rho_0_75', 'gdf_rho_0_7', 'gdf_rho_0_649', 'gdf_rho_0_599', 'gdf_rho_0_549', 
    'gdf_rho_0_499', 'gdf_rho_0_449', 'gdf_rho_0_399', 'gdf_rho_0_349', 'gdf_rho_0_299', 
    'gdf_rho_0_249', 'gdf_rho_0_199', 'gdf_rho_0_149', 'gdf_rho_0_099', 'gdf_rho_0_049'
    ]
connection_link_var = repo_path_r_string + r'\03_simulated_data\simulated_data.sqlite'

In [4]:
dict_of_gdfs = {}
for i in df_names_list:
        gdf = import_spatial_point_data_frame(
                connection_link_lnx = repo_path_unix+"/03_simulated_data/simulated_data.sqlite",
                layer_name =i,
                crs_param = 3857
                                        )
        df = import_non_spatial_data_frame(
                connection_link = connection_link_var,
                df_to_return_name =  f'{i}_sp'
                                        )
        df2 = df[['unit_id']+[c for c in df.columns.tolist() if 'spatial_predictor' in c]].copy()
        gdf2 = gdf.merge(df2, on = ['unit_id'])
        dict_of_gdfs[i] = gdf2

In [5]:
gdf_pre =dict_of_gdfs['gdf_rho_0_75'].copy()
gdf = gdf_pre.drop(columns = [c for c in dict_of_gdfs['gdf_rho_0_75'].columns.tolist() if 'spatial_predictor' in c]).copy()
df = import_non_spatial_data_frame(
                connection_link = connection_link_var,
                df_to_return_name =  'gdf_rho_0_75_sp_t_dr5'
                                        )

df2 = df[['unit_id']+[c for c in df.columns.tolist() if 'spatial_predictor' in c]].copy()
gdf2 = gdf.merge(df2, on = ['unit_id'])

In [6]:
dict_of_gdfs['gdf_rho_0_75'].isna().any().any()

np.False_

In [7]:
dict_of_gdfs.keys()

dict_keys(['gdf_rho_0_75', 'gdf_rho_0_7', 'gdf_rho_0_649', 'gdf_rho_0_599', 'gdf_rho_0_549', 'gdf_rho_0_499', 'gdf_rho_0_449', 'gdf_rho_0_399', 'gdf_rho_0_349', 'gdf_rho_0_299', 'gdf_rho_0_249', 'gdf_rho_0_199', 'gdf_rho_0_149', 'gdf_rho_0_099', 'gdf_rho_0_049'])

# Modelling causal inference

## Single df

### Data preparation

In [12]:
gdf2

,unit_id,t,propensity,t_tot,t_tot_cat,t_tot_cat_underestim,t_tot_cat_overerestim,odr_1,odr_2,odr_3,...,geometry,spatial_predictor_100_1,spatial_predictor_100_3,spatial_predictor_100_4,spatial_predictor_100_2,spatial_predictor_100_6,spatial_predictor_100_5,spatial_predictor_100_9,spatial_predictor_500_9,spatial_predictor_100_10
0,1000,0,4.254438,0,control,control,control,0,0,0,...,POINT (0 0),-0.012927,0.010595,0.025685,-0.004391,0.017704,-0.015820,-0.022337,0.020665,-0.039483
1,1001,0,2.048454,0,control,control,control,0,0,0,...,POINT (0 100),-0.018153,0.015350,0.036128,-0.006397,0.025482,-0.023075,-0.033027,0.024074,-0.053469
2,1002,0,1.078793,0,control,control,control,0,0,0,...,POINT (0 200),-0.021105,0.018410,0.041466,-0.007911,0.030127,-0.027608,-0.039891,0.027345,-0.056255
3,1003,0,-2.519299,0,control,control,control,0,0,0,...,POINT (0 300),-0.022892,0.020462,0.043284,-0.009118,0.032335,-0.030302,-0.043785,0.030559,-0.049756
4,1004,0,1.401347,0,control,control,control,0,0,0,...,POINT (0 400),-0.024033,0.021902,0.042545,-0.010160,0.032619,-0.031721,-0.045301,0.033249,-0.036231
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
795,1795,0,0.546255,0,control,control,control,0,0,0,...,POINT (3900 1500),0.090029,0.034072,0.015740,-0.083012,0.008130,0.041915,0.003435,0.047070,-0.002933
796,1796,0,0.217175,0,control,control,control,0,0,0,...,POINT (3900 1600),0.102460,0.033038,0.026323,-0.115110,-0.009900,0.063532,-0.000478,0.057962,-0.001135
797,1797,0,-0.753369,0,control,control,control,0,0,0,...,POINT (3900 1700),0.119752,0.033995,0.038097,-0.152442,-0.028509,0.089029,-0.003795,0.067823,0.000879
798,1798,0,1.541784,0,control,control,control,0,0,0,...,POINT (3900 1800),0.144678,0.037568,0.050946,-0.196466,-0.046410,0.117683,-0.006542,0.077051,0.002785


In [22]:
X_ns = gdf2[["c1","c2",
         ]].values
X = gdf2[["c1","c2",
         'spatial_predictor_100_1',
    'spatial_predictor_100_3',
    'spatial_predictor_100_4',
    'spatial_predictor_100_2',
    'spatial_predictor_100_6',
    'spatial_predictor_100_5',
    'spatial_predictor_100_9',
    'spatial_predictor_500_9',
    'spatial_predictor_100_10'
         ]].values
y = gdf2["y"].values.reshape(-1,1)
T = gdf2["t"].values
T_tot = gdf2["t_tot"].values
Xdf = gdf2[['t','odr_1','odr_2','odr_3','odr_4', "c1","c2",
         'spatial_predictor_100_1',
    'spatial_predictor_100_3',
    'spatial_predictor_100_4',
    'spatial_predictor_100_2',
    'spatial_predictor_100_6',
    'spatial_predictor_100_5',
    'spatial_predictor_100_9',
    'spatial_predictor_500_9',
    'spatial_predictor_100_10'
         ]]
Xdf2 = gdf2[['t','odr_1','odr_2','odr_3','odr_4', "c1","c2",
         #"C3","Cs"
         ]]
X_l = sm.add_constant(Xdf)
X_l2 = sm.add_constant(Xdf2)
y_l = gdf2["y_ns"]
Y = gdf2['y'].values
T_M= gdf2['t_tot_cat'].values
T_M= T_M.astype(str)

gdf3 = gdf2.copy()
gdf3 = gdf3.rename(columns = {'t': 'treated_inner_ring',
                              'odr_1':'treated_outer_ring1', 
                              'odr_2':'treated_outer_ring2', 
                              'odr_3':'treated_outer_ring3', 
                              'odr_4':'treated_outer_ring4'
                              })
T_mult = gdf3[[ 
         'treated_inner_ring', 'treated_outer_ring1','treated_outer_ring2','treated_outer_ring3','treated_outer_ring4']].to_numpy()

gdf2_22 = gdf.copy()
gdf2_22["x"] = gdf2_22.geometry.x.astype(int)
gdf2_22["y"] = gdf2_22.geometry.y.astype(int)

n = len(gdf2_22)

coords = np.array(
        list(zip(gdf2_22["x"], gdf2_22["y"]))
    )


W_mat = DistanceBand(
        coords,
        threshold=100 + 1,
        binary=True,
        silence_warnings=True
    ) 

### Multi treatment

#### With spatial regressors

##### S learner

In [30]:
RF_s_learner = BaseSRegressor(RandomForestRegressor(n_estimators=160, max_depth=10, random_state=42), control_name = 'control')
ate_slearn_multT, lb_slearn_multT, ub_slearn_multT = RF_s_learner.estimate_ate(X, T_M, Y, return_ci = True)
ite_slearn_multT = RF_s_learner.fit_predict(X, T_M, Y)
ITE_df_slearn_multT = pd.DataFrame(ite_slearn_multT).rename(columns = {0:'treated_inner_ring', 1:'treated_outer_ring1', 
                                             2:'treated_outer_ring2', 3:'treated_outer_ring3',
                                             4:'treated_outer_ring4'})
ITE_df_slearn_multT['treated'] = T_M
ITE_df_slearn_multT['ITE_real'] = gdf2['tau']
ITE_df_slearn_multT['Y'] = Y
ITE_df_slearn_multT

,treated_inner_ring,treated_outer_ring1,treated_outer_ring2,treated_outer_ring3,treated_outer_ring4,treated,ITE_real,Y
0,0.091079,0.236914,-0.007305,0.007885,0.000000,control,0.0,3.583302
1,1.427213,1.132947,0.039986,0.007854,-0.009534,control,0.0,1.664774
2,1.270539,0.518246,0.006733,0.030715,-0.010028,control,0.0,0.924681
3,2.791680,1.923822,0.040104,0.002627,-0.010918,control,0.0,-1.424234
4,2.087766,1.014219,0.138980,0.032181,-0.015451,control,0.0,0.939380
...,...,...,...,...,...,...,...,...
795,1.075604,0.201128,0.010883,-0.011217,-0.008061,control,0.0,0.515627
796,2.213119,1.661387,-0.003793,0.041582,-0.016875,control,0.0,-0.243606
797,2.316802,0.133477,-0.022041,0.007760,-0.025951,control,0.0,-0.990687
798,1.463977,1.059068,0.034945,0.000000,-0.022462,control,0.0,0.898532


In [31]:
df_effectbase = pd.DataFrame({
                'ring':['treated_inner_ring', 'treated_outer_ring1', 'treated_outer_ring2','treated_outer_ring3','treated_outer_ring4'],
              'true_effect':[gdf2[gdf2['t'] == 1]['tau'].mean(), gdf2[gdf2['odr_1'] == 1]['tau'].mean(), 
                             gdf2[gdf2['odr_2'] == 1]['tau'].mean(), gdf2[gdf2['odr_3'] == 1]['tau'].mean(), 
                               gdf2[gdf2['odr_4'] == 1]['tau'].mean()],
              })
df_multi_treatment_effects_Slearn = make_treatment_effects_df(ITE_df_slearn_multT,  [ 'treated_inner_ring','treated_outer_ring1',
             'treated_outer_ring2','treated_outer_ring3', 'treated_outer_ring4'],'mSlearn_sp', treated_col='treated')
df_multi_effect_pre1 = df_effectbase.merge(df_multi_treatment_effects_Slearn, on = 'ring', how= 'left')
df_multi_effect_pre1

,ring,true_effect,att_mSlearn_sp,se_mSlearn_sp
0,treated_inner_ring,1.500000,1.553143,0.072211
1,treated_outer_ring1,1.049929,0.684153,0.071589
2,treated_outer_ring2,0.535380,0.034562,0.009169
3,treated_outer_ring3,0.123083,0.017301,0.002495
4,treated_outer_ring4,0.013995,-0.003085,0.003673


##### Linear DML

In [32]:
est_LDML = LinearDML(model_y=RandomForestRegressor(n_estimators=100, max_depth=3, min_samples_leaf=20),
                model_t=MultiOutputRegressor(RandomForestRegressor(n_estimators=100,
                                                                       max_depth=3,
                                                                       min_samples_leaf=20)),
                featurizer=PolynomialFeatures(degree=2, include_bias=False),
                cv=None)

est_LDML.fit(Y, T_mult, X=X, 
        #W=W
        )
te_pred_LDML = est_LDML.const_marginal_effect(X)

ITE_LDML= pd.DataFrame(te_pred_LDML).rename(columns = {0:'treated_inner_ring', 1:'treated_outer_ring1', 
                                             2:'treated_outer_ring2', 3:'treated_outer_ring3',
                                             4:'treated_outer_ring4'})
ITE_LDML['treated'] = T_M
df_multi_treatment_effects_LDML = make_treatment_effects_df(ITE_LDML,  ['treated_inner_ring','treated_outer_ring1',
             'treated_outer_ring2','treated_outer_ring3', 'treated_outer_ring4'],'mLDML_sp', treated_col='treated')
df_multi_effect_pre2 = df_multi_effect_pre1.merge(df_multi_treatment_effects_LDML, on = 'ring', how= 'left')
df_multi_effect_pre2 

,ring,true_effect,att_mSlearn_sp,se_mSlearn_sp,att_mLDML_sp,se_mLDML_sp
0,treated_inner_ring,1.500000,1.553143,0.072211,-5.475922,1.029731
1,treated_outer_ring1,1.049929,0.684153,0.071589,-1.089245,0.668674
2,treated_outer_ring2,0.535380,0.034562,0.009169,-0.550956,0.374371
3,treated_outer_ring3,0.123083,0.017301,0.002495,0.135284,0.161985
4,treated_outer_ring4,0.013995,-0.003085,0.003673,-0.157875,0.135401


##### Causal forest

In [33]:
est2 = CausalForestDML(model_y=RandomForestRegressor(n_estimators=100, max_depth=3, min_samples_leaf=20),
                       model_t=MultiOutputRegressor(RandomForestRegressor(n_estimators=100,
                                                                              max_depth=3,
                                                                              min_samples_leaf=20)),
                       cv=None,
                       criterion='mse', n_estimators=1000,
                       min_samples_leaf=10,
                       min_impurity_decrease=0.001,
                       random_state=123)

est2.tune(Y, T_mult, X=X, 
          #W=W
          )
est2.fit(Y, T_mult, X=X, 
         #W=W
         )

te_pred2 = est2.const_marginal_effect(X)
ITE_CF=pd.DataFrame(te_pred2).rename(columns = {0:'treated_inner_ring', 1:'treated_outer_ring1', 
                                             2:'treated_outer_ring2', 3:'treated_outer_ring3',
                                             4:'treated_outer_ring4'})
ITE_CF['treated'] = T_M
df_multi_treatment_effects_CF = make_treatment_effects_df(ITE_CF,  ['treated_inner_ring','treated_outer_ring1',
             'treated_outer_ring2','treated_outer_ring3', 'treated_outer_ring4'],'mCF_sp', treated_col='treated')
df_multi_effect_pre3 = df_multi_effect_pre2.merge(df_multi_treatment_effects_CF, on = 'ring', how= 'left')
df_multi_effect_pre3 

,ring,true_effect,att_mSlearn_sp,se_mSlearn_sp,att_mLDML_sp,se_mLDML_sp,att_mCF_sp,se_mCF_sp
0,treated_inner_ring,1.500000,1.553143,0.072211,-5.475922,1.029731,1.025987,0.018796
1,treated_outer_ring1,1.049929,0.684153,0.071589,-1.089245,0.668674,0.821603,0.024904
2,treated_outer_ring2,0.535380,0.034562,0.009169,-0.550956,0.374371,0.028879,0.040458
3,treated_outer_ring3,0.123083,0.017301,0.002495,0.135284,0.161985,-0.022223,0.024872
4,treated_outer_ring4,0.013995,-0.003085,0.003673,-0.157875,0.135401,-0.199969,0.013707


##### Cross sectional DiD

In [34]:
model_cs_did_sp_pred = sm.OLS(y_l, X_l)
results_cs_did_sp_pred = model_cs_did_sp_pred.fit()

coef_df_cs_did_sp_pred = pd.DataFrame({
    "ring": results_cs_did_sp_pred.params.index,
    "att_mDiD_sp_pred": results_cs_did_sp_pred.params.values,
    "se_mDiD_sp_pred": results_cs_did_sp_pred.bse.values
})
coef_df_cs_did_sp_pred = coef_df_cs_did_sp_pred[coef_df_cs_did_sp_pred['ring'].isin(['t', 
                                                            'odr_1', 'odr_2',
                                                            'odr_3', 'odr_4'])].copy()

coef_df_cs_did_sp_pred['ring'] = coef_df_cs_did_sp_pred['ring'].replace({'t':'treated_inner_ring', 
                                                                        'odr_1':'treated_outer_ring1', 
                                                                        'odr_2':'treated_outer_ring2',
                                                                        'odr_3':'treated_outer_ring3',
                                                                          'odr_4':'treated_outer_ring4'})
df_multi_effect_pre4 = df_multi_effect_pre3.merge(coef_df_cs_did_sp_pred, on = 'ring', how= 'left')
df_multi_effect_pre4

,ring,true_effect,att_mSlearn_sp,se_mSlearn_sp,att_mLDML_sp,se_mLDML_sp,att_mCF_sp,se_mCF_sp,att_mDiD_sp_pred,se_mDiD_sp_pred
0,treated_inner_ring,1.500000,1.553143,0.072211,-5.475922,1.029731,1.025987,0.018796,1.274743,0.141234
1,treated_outer_ring1,1.049929,0.684153,0.071589,-1.089245,0.668674,0.821603,0.024904,0.976404,0.165074
2,treated_outer_ring2,0.535380,0.034562,0.009169,-0.550956,0.374371,0.028879,0.040458,0.220732,0.155927
3,treated_outer_ring3,0.123083,0.017301,0.002495,0.135284,0.161985,-0.022223,0.024872,0.163043,0.142451
4,treated_outer_ring4,0.013995,-0.003085,0.003673,-0.157875,0.135401,-0.199969,0.013707,-0.048582,0.134652


In [42]:
variables

['CONSTANT', 't', 'odr_1', 'odr_2', 'odr_3', 'odr_4', 'c1', 'c2', 'W_y', 'rho']

In [43]:
len(ses)

9

In [36]:
model_sdm = ML_Lag(
    y,
    X_l2,
    w=W_mat,              
    slx_lags=0,       
    name_y="y",
    # name_x=['t','odr_1','odr_2','odr_3','odr_4', "c1","c2"
    #         ]
)

coefs = model_sdm.betas.flatten()

# błędy standardowe
ses = np.sqrt(np.diag(model_sdm.vm))

# nazwy zmiennych
variables = model_sdm.name_x.copy()

# dodanie spatial lag parametru rho
variables = variables + ["rho"]

# dataframe
coef_df = pd.DataFrame({
    "variable": variables,
    "coef": coefs,
    "se": ses
})
coef_df

ML_Lag


ValueError: All arrays must be of the same length

#### Without spatial regressors

##### S learner

In [ ]:
RF_s_learner_nsp = BaseSRegressor(RandomForestRegressor(n_estimators=160, max_depth=10, random_state=42), control_name = 'control')
ate_slearn_multT_nsp, lb_slearn_multT_nsp, ub_slearn_multT_nsp = RF_s_learner.estimate_ate(X_ns, T_M, Y, return_ci = True)
ite_slearn_multT_nsp = RF_s_learner_nsp.fit_predict(X_ns, T_M, Y)
ITE_df_slearn_multT_nsp = pd.DataFrame(ite_slearn_multT_nsp).rename(columns = {0:'treated_inner_ring', 1:'treated_outer_ring1', 
                                             2:'treated_outer_ring2', 3:'treated_outer_ring3',
                                             4:'treated_outer_ring4'})
ITE_df_slearn_multT_nsp['treated'] = T_M
df_multi_treatment_effects_Slearn_nsp = make_treatment_effects_df(ITE_df_slearn_multT_nsp,  [ 'treated_inner_ring','treated_outer_ring1',
             'treated_outer_ring2','treated_outer_ring3', 'treated_outer_ring4'],'mSlearn_nsp', treated_col='treated')
df_multi_effect_pre4 = df_multi_effect_pre3.merge(df_multi_treatment_effects_Slearn_nsp, on = 'ring', how= 'left')
df_multi_effect_pre4

##### Linear DML

In [ ]:
est_LDML_nsp = LinearDML(model_y=RandomForestRegressor(n_estimators=100, max_depth=3, min_samples_leaf=20),
                model_t=MultiOutputRegressor(RandomForestRegressor(n_estimators=100,
                                                                       max_depth=3,
                                                                       min_samples_leaf=20)),
                featurizer=PolynomialFeatures(degree=2, include_bias=False),
                cv=None)

est_LDML_nsp.fit(Y, T_mult, X=X_ns, 
        #W=W
        )
te_pred_LDML_nsp = est_LDML_nsp.const_marginal_effect(X_ns)

ITE_LDML_nsp= pd.DataFrame(te_pred_LDML_nsp).rename(columns = {0:'treated_inner_ring', 1:'treated_outer_ring1', 
                                             2:'treated_outer_ring2', 3:'treated_outer_ring3',
                                             4:'treated_outer_ring4'})
ITE_LDML_nsp['treated'] = T_M
df_multi_treatment_effects_LDML_nsp = make_treatment_effects_df(ITE_LDML_nsp,  ['treated_inner_ring','treated_outer_ring1',
             'treated_outer_ring2','treated_outer_ring3', 'treated_outer_ring4'],'mLDML_nsp', treated_col='treated')
df_multi_effect_pre5 = df_multi_effect_pre4.merge(df_multi_treatment_effects_LDML_nsp, on = 'ring', how= 'left')
df_multi_effect_pre5 

##### Causal forest

In [ ]:
est2_cf_nsp = CausalForestDML(model_y=RandomForestRegressor(n_estimators=100, max_depth=3, min_samples_leaf=20),
                       model_t=MultiOutputRegressor(RandomForestRegressor(n_estimators=100,
                                                                              max_depth=3,
                                                                              min_samples_leaf=20)),
                       cv=None,
                       criterion='mse', n_estimators=1000,
                       min_samples_leaf=10,
                       min_impurity_decrease=0.001,
                       random_state=123)

est2_cf_nsp.tune(Y, T_mult, X=X_ns, 
          #W=W
          )
est2_cf_nsp.fit(Y, T_mult, X=X_ns, 
         #W=W
         )

te_pred2_nsp = est2_cf_nsp.const_marginal_effect(X_ns)
ITE_CF_nsp=pd.DataFrame(te_pred2_nsp).rename(columns = {0:'treated_inner_ring', 1:'treated_outer_ring1', 
                                             2:'treated_outer_ring2', 3:'treated_outer_ring3',
                                             4:'treated_outer_ring4'})
ITE_CF_nsp['treated'] = T_M
df_multi_treatment_effects_CF_nsp = make_treatment_effects_df(ITE_CF_nsp,  ['treated_inner_ring','treated_outer_ring1',
             'treated_outer_ring2','treated_outer_ring3', 'treated_outer_ring4'],'mCF_nsp', treated_col='treated')
df_multi_effect_final = df_multi_effect_pre5.merge(df_multi_treatment_effects_CF_nsp, on = 'ring', how= 'left')
df_multi_effect_final 

In [ ]:
att_models = {
    "mSlearn_sp": ("att_mSlearn_sp", "se_mSlearn_sp"),
    #"mLDML_sp": ("att_mLDML_sp", "se_mLDML_sp"),
    "mCF_sp": ("att_mCF_sp", "se_mCF_sp"),
    "mSlearn_nsp": ("att_mSlearn_nsp", "se_mSlearn_nsp"),
    #"mLDML_nsp": ("att_mLDML_nsp", "se_mLDML_nsp"),
    "mCF_nsp": ("att_mCF_nsp", "se_mCF_nsp")
}
for ring in df_multi_effect_final['ring'].unique().tolist():
    plot_att_row(
        df=df_multi_effect_final,
        ring_name=ring,
        true_effect_col="true_effect",
        att_dict=att_models
    )

#### Results visualization

In [ ]:
att_models = {
    "mSlearn_sp": ("att_Slearn", "se_Slearn"),
    #"LDML": ("att_LDML", "se_LDML"),
    "mCF_sp": ("att_mCF_sp", "se_mCF_sp"),
    "mSlearn_nsp": ("att_mSlearn_nsp", "se_mSlearn_nsp"),
    "mLDML_nsp": ("att_mLDML_nsp", "se_mLDML_nsp"),
    "mCF_nsp": ("att_mCF_nsp", "se_mCF_nsp")
}

plot_att_row(
    df=df_multi_effect_final,
    ring_name="treated_inner_ring",
    true_effect_col="true_effect",
    att_dict=att_models
)

### Single treatment

#### S learner

#### Causal forest

## Loop

In [ ]:
features_non_spatial = ['c1', 'c2']
list_of_effects_ik_dfs = []

for i in df_names_list:

    features_spatial = features_non_spatial+[c for c in dict_of_gdfs[i].columns.tolist() if 'spatial_predictor' in c]
    X_non_sp = dict_of_gdfs[i][features_non_spatial].values
    X_sp = dict_of_gdfs[i][features_spatial].values
    Y =  dict_of_gdfs[i]['y'].values
    T =  dict_of_gdfs[i]['t'].values

    dict_of_forests = {
        'cf_sp': X_sp,
        'cf_nsp': X_non_sp
    }
    gdf_temp = dict_of_gdfs[i].copy()
    list_of_effects_k_dfs = []
    for k in dict_of_forests.keys():
        est = CausalForestDML(model_t=RandomForestRegressor(),
                        model_y=RandomForestRegressor(),
                        n_estimators=180, #min_samples_leaf=5,
                        max_depth=10,
                        #verbose=0, 
                        random_state=42)
        est.fit(Y, T, X=dict_of_forests[k])
        tau_hat_cf = est.effect(dict_of_forests[k])
        gdf_temp[f'ITE_{k}'] = tau_hat_cf
        tau_hat_cf_treated = gdf_temp[gdf_temp['t']==1][f'ITE_{k}'].copy()
        att = tau_hat_cf_treated.mean()
        se_att = tau_hat_cf_treated.std(ddof=1) / np.sqrt(len(tau_hat_cf_treated))
        ci_low_att, ci_high_att = att - 1.96 * se_att, att + 1.96 * se_att
        df_result_temp = pd.DataFrame({'model_name': [f'{k}'],'ci_low_att':[ci_low_att], 'att':[att], 'ci_high_att':[ci_high_att], 'rho': float('0.'+i[10:])})
        list_of_effects_k_dfs.append(df_result_temp)
    df_cf_att_i = pd.concat(list_of_effects_k_dfs)
    list_of_effects_ik_dfs.append(df_cf_att_i)
df_cf_att = pd.concat(list_of_effects_ik_dfs)

# Results visualization

In [ ]:
df_cf_att

In [ ]:
df_sp = df_cf_att[df_cf_att['model_name'] == 'cf_sp'].copy()
df_nsp = df_cf_att[df_cf_att['model_name'] == 'cf_nsp'].copy()

# lekkie przesunięcie na osi X
offset = 0.01
df_sp['rho_shift'] = df_sp['rho'] - offset
df_nsp['rho_shift'] = df_nsp['rho'] + offset

fig, ax = plt.subplots(figsize=(8, 5))

# --- cf_sp (zielony) ---
ax.scatter(df_sp['rho_shift'], df_sp['att'], color='green', label='cf_sp', zorder=3)

ax.vlines(
    x=df_sp['rho_shift'],
    ymin=df_sp['ci_low_att'],
    ymax=df_sp['ci_high_att'],
    color='green',
    linewidth=3
)

# --- cf_nsp (niebieski) ---
ax.scatter(df_nsp['rho_shift'], df_nsp['att'], color='blue', label='cf_nsp', zorder=3)

ax.vlines(
    x=df_nsp['rho_shift'],
    ymin=df_nsp['ci_low_att'],
    ymax=df_nsp['ci_high_att'],
    color='blue',
    linewidth=3
)

# pozioma linia odniesienia
ax.axhline(y=1.5, color='red', linestyle='--')

# opisy
ax.set_xlabel('rho')
ax.set_ylabel('ATT')
ax.legend()

plt.tight_layout()
plt.show()